### Обучающая и тестовая выборки

- В качестве обучающей выборки будем использовать наборы, полученные при помощи train_test_split из scikit-learn.
- Конечно, алгоритм может работать с многомерным пространством, но для удобства демонстраиции алгоритма мы используем двухмерный набор данных.
- Если разные признаки имеют сильно отличающиеся диапазоны значений, то применяют масштабирование исходных данных.

In [267]:
# Установим необходимые библиотеки (раскоментировать)
#%pip install numpy pandas sklearn

In [268]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [269]:
data_full = pd.read_csv('data/ramen-ratings.csv', sep=",")
data_full

,Review #,Brand,Variety,Style,Country,Stars,Top Ten
0,2580,New Touch,T's Restaurant Tantanmen,Cup,Japan,3.75,NaN
1,2579,Just Way,Noodles Spicy Hot Sesame Spicy Hot Sesame Guan...,Pack,Taiwan,1,NaN
2,2578,Nissin,Cup Noodles Chicken Vegetable,Cup,USA,2.25,NaN
3,2577,Wei Lih,GGE Ramen Snack Tomato Flavor,Pack,Taiwan,2.75,NaN
4,2576,Ching's Secret,Singapore Curry,Pack,India,3.75,NaN
...,...,...,...,...,...,...,...
2575,5,Vifon,"Hu Tiu Nam Vang [""Phnom Penh"" style] Asian Sty...",Bowl,Vietnam,3.5,NaN
2576,4,Wai Wai,Oriental Style Instant Noodles,Pack,Thailand,1,NaN
2577,3,Wai Wai,Tom Yum Shrimp,Pack,Thailand,2,NaN
2578,2,Wai Wai,Tom Yum Chili Flavor,Pack,Thailand,2,NaN


Удаление ненужных столбцов

In [270]:
data_full = data_full.drop(columns=['Review #', 'Top Ten'])

Проверим датасет на пустые значения. 

In [271]:
# проверим есть ли пропущенные значения
data_full.isnull().sum()

Brand      0
Variety    0
Style      2
Country    0
Stars      0
dtype: int64

Обработка Style

In [272]:
data_full['Style'] = data_full['Style'].fillna("Unknown").astype(str)

Повторная проверка на пустые значения

In [273]:
data_full.isnull().sum()

Brand      0
Variety    0
Style      0
Country    0
Stars      0
dtype: int64

Заметим, что Style является категориальным признаком, рассмотрим уникальные значения для него:

In [274]:
np.unique(data_full[['Style']])

array(['Bar', 'Bowl', 'Box', 'Can', 'Cup', 'Pack', 'Tray', 'Unknown'],
      dtype=object)

Обработка Stars

In [275]:
data_full['Stars'] = pd.to_numeric(data_full['Stars'], errors='coerce')

# Замена NaN в 'Stars' на медианное значение
median_stars = data_full['Stars'].median()
data_full['Stars'] = data_full['Stars'].fillna(median_stars)


In [276]:
# Преобразование с корректными границами
bins = [0, 1.5, 3.0, 4.0, 5.0] 
labels = ['very_low', 'low', 'medium', 'high']

# Преобразование
data_full['Stars'] = pd.cut(
    data_full['Stars'].astype(float),
    bins=bins,
    labels=labels,
    include_lowest=True
)
data_full.isnull().sum()

Brand      0
Variety    0
Style      0
Country    0
Stars      0
dtype: int64

Закодируем остальные строковые показатели при помощи простого LabelEncoder.

In [277]:
categorical_features = ['Brand', 'Variety', 'Country', 'Style']

# Инициализация LabelEncoder
label_encoders = {}

In [278]:
# Перекодировка категориальных признаков
for feature in categorical_features:
    le = LabelEncoder()
    data_full[feature] = le.fit_transform(data_full[feature])
    label_encoders[feature] = le

Разделим выборку на тестовую и тренировочную.

In [279]:
X = data_full.drop(columns=['Stars'])
y = data_full['Stars']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

In [280]:
data_train_to_csv=X_train
data_train_to_csv['Stars']=y_train
data_train_to_csv

,Brand,Variety,Style,Country,Stars
1710,282,1279,5,17,medium
2050,163,98,1,33,medium
2364,160,898,4,35,low
361,63,1436,4,0,very_low
2023,23,2153,5,34,medium
...,...,...,...,...,...
1638,123,673,5,5,medium
1095,195,2095,5,35,high
1130,192,1043,5,29,high
1294,23,2152,5,34,medium


In [281]:
data_test_to_csv=X_test
data_test_to_csv['Stars']=y_test
data_test_to_csv

,Brand,Variety,Style,Country,Stars
809,116,2165,1,5,medium
2356,157,869,1,33,low
761,350,1648,1,33,high
318,56,547,4,35,high
961,192,390,5,29,medium
...,...,...,...,...,...
1056,192,389,5,29,medium
1027,159,1301,5,19,medium
1010,160,181,1,18,high
25,249,1986,1,30,high


Для второй тестовой выборки возьмем все данные датасета.

In [282]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99,
                                                    train_size=0.001, 
                                                    random_state=42)
data_test_to_csv2=X_test
data_test_to_csv2['Stars']=y_test
data_test_to_csv2

,Brand,Variety,Style,Country,Stars
809,116,2165,1,5,medium
2356,157,869,1,33,low
761,350,1648,1,33,high
318,56,547,4,35,high
961,192,390,5,29,medium
...,...,...,...,...,...
1806,110,1762,5,17,low
975,192,582,5,14,medium
2047,208,729,5,30,high
1082,249,1985,5,30,medium


Запишем полученные датасеты в .csv и запишем в новые переменные (для чистоты эксперимента).

In [283]:
pd.DataFrame.to_csv(data_test_to_csv, './data/ramen_test_multiclass.csv', sep=",", index=False)
pd.DataFrame.to_csv(data_train_to_csv, './data/ramen_train_multiclass.csv', sep=",", index=False)
pd.DataFrame.to_csv(data_test_to_csv2, './data/ramen_test_full_multiclass.csv', sep=",", index=False)